In [12]:
import torch
import ray
import os
import pandas as pd
import numpy as np
import re
from scipy.stats import spearmanr

from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# Define which slides you want to process here
slide_path = "/mnt/projects/ri_scale/privagams/prostate_stains"

# Code for similarity taken from silde_embeddings.py

In [2]:
def load_parquet(path): 
    data = pd.read_parquet(path)
    return data

def load_embeddings(slide_path, file_name):
    dirs_to_process = sorted([
        d for d in os.listdir(slide_path) 
        if os.path.isdir(os.path.join(slide_path, d)) and not d.startswith(".")
    ])
    embeddings: List[torch.Tensor] = []
    labels = []

    for dir in dirs_to_process:
        embeddings_path = os.path.join(slide_path, dir, file_name)
        tensor = torch.tensor(load_parquet(embeddings_path).embedding[0], dtype=torch.float32)
        embeddings.append(tensor)
        labels.append(dir)
    
    emb_matrix = torch.stack(embeddings, dim=0)

    return (emb_matrix, labels)

def save_simmilarity(sim_matrix, labels, name):

    # making flash as rows and midi as cols
    keep_rows = [label.startswith("FLASH") for label in labels]
    keep_cols = [label.startswith("MIDI") for label in labels]
    x_labels = [label for label in labels if label.startswith("MIDI")]
    y_labels = [label for label in labels if label.startswith("FLASH")]

    sim_matrix = sim_matrix[keep_rows, :]
    sim_matrix = sim_matrix[:, keep_cols]

    diagonal_values = torch.diag(sim_matrix)
    row_max_values = torch.max(sim_matrix, dim=1).values
    is_diagonal_max = (diagonal_values == row_max_values)
    count = torch.sum(is_diagonal_max).item() 

    #print(f"Number of MIDI/FLASH matches for {name} is {count}, R_s= {count/len(x_labels)}")
    #print(f"length of {name} is {len(x_labels)}")
    return count

def cos_simmilarity(slide_path, file_name):
    emb_matrix, labels = load_embeddings(slide_path, file_name)
    X_norm = F.normalize(emb_matrix, p=2, dim=1)
    cosine_sim = X_norm @ X_norm.T
    return save_simmilarity(cosine_sim, labels, f"cos_{file_name}")

def l1_simmilarity(slide_path, file_name):
    emb_matrix, labels = load_embeddings(slide_path, file_name)
    l1_distance_matrix = torch.cdist(emb_matrix, emb_matrix, p=1)

    D_min = l1_distance_matrix.min()
    D_max = l1_distance_matrix.max()

    # 2. Normalizace matice na rozsah [0, 1]
    # (Odečteme minimum a vydělíme rozsahem)
    D_range = D_max - D_min
    # Ošetření případu, kdy D_range je nula (např. matice plná stejných hodnot)
    if D_range == 0:
        similarity_matrix = torch.ones_like(l1_distance_matrix)
    else:
        D_norm = (l1_distance_matrix - D_min) / D_range
        
        # 3. Inverze (Odečtení od 1)
        similarity_matrix = 1 - D_norm

    return save_simmilarity(similarity_matrix, labels, f"L1_{file_name}")

def l2_simmilarity(slide_path, file_name):
    emb_matrix, labels = load_embeddings(slide_path, file_name)
    l2_distance_matrix = torch.cdist(emb_matrix, emb_matrix, p=2)

    D_min = l2_distance_matrix.min()
    D_max = l2_distance_matrix.max()

    # 2. Normalizace matice na rozsah [0, 1]
    # (Odečteme minimum a vydělíme rozsahem)
    D_range = D_max - D_min
    # Ošetření případu, kdy D_range je nula (např. matice plná stejných hodnot)
    if D_range == 0:
        similarity_matrix = torch.ones_like(l2_distance_matrix)
    else:
        D_norm = (l2_distance_matrix - D_min) / D_range
        
        # 3. Inverze (Odečtení od 1)
        similarity_matrix = 1 - D_norm

    return save_simmilarity(similarity_matrix, labels, f"L2_{file_name}")

# Code for playing directly with tiles embeddings

In [13]:
def load_parquet(path): 
    data = pd.read_parquet(path)
    return data

def load_tiles_embeddings(slide_path):
    dirs_to_process = sorted([
        d for d in os.listdir(slide_path) 
        if os.path.isdir(os.path.join(slide_path, d)) and not d.startswith(".")
    ])
    embeddings: List[List[torch.Tensor]] = []
    labels = []

    for dir in dirs_to_process:
        embeddings_path = os.path.join(slide_path, dir, "tiles.parquet")
        embedding_array = np.array(load_parquet(embeddings_path).embedding.tolist())
        embedding_matrix = torch.tensor(embedding_array, dtype=torch.float32)
        embeddings.append(embedding_matrix)
        labels.append(dir)
    

    return (embeddings, labels)

#### getting tiles embedding for each slide in slide_path

In [14]:
matrix, labels = load_tiles_embeddings(slide_path)

#### generating mean of all tiles embedding to act as slide embedding

In [15]:
for i in range(len(labels)):
    wsi_embedding = torch.mean(matrix[i], dim=0)
    label = labels[i]
    output_df = pd.DataFrame({
        'slide_id': label,
        'embedding': [wsi_embedding.tolist()],
    })
    output_df.to_parquet(f"{slide_path}/{label}/slide_mean.parquet", index=False)

### Statistical pooling

In [16]:
for i in range(len(labels)):
    mean_vector = torch.mean(matrix[i], dim=0)
    std_dev_vector = torch.std(matrix[i], dim=0)
    wsi_embedding = torch.cat((mean_vector, std_dev_vector), dim=0)

    label = labels[i]
    output_df = pd.DataFrame({
        'slide_id': label,
        'embedding': [wsi_embedding.tolist()],
    })
    output_df.to_parquet(f"{slide_path}/{label}/slide_statistic.parquet", index=False)



In [20]:
slide_paths = [
   "/mnt/projects/ri_scale/privagams/breast_rgb_filter",
   "/mnt/projects/ri_scale/privagams/colon_rgb_filter",
   "/mnt/projects/ri_scale/privagams/prostate_rgb_filter",
]

slide_paths_stains = [
   "/mnt/projects/ri_scale/privagams/breast_stains",
   "/mnt/projects/ri_scale/privagams/colon_stains",
   "/mnt/projects/ri_scale/privagams/prostate_stains",
]

embeddings_files = [
   "slide.parquet",
   "slide_mean.parquet",
   "slide_statistic.parquet"
]

for file in embeddings_files:
   cos = 0
   l1 = 0
   l2 = 0
   for slide_path in slide_paths_stains:
      cos += cos_simmilarity(slide_path, file)
      l1 += l1_simmilarity(slide_path, file)
      l2 += l2_simmilarity(slide_path, file)
   
   print()
   print(file)
   length = 30
   print(f"length={length}\ncos={cos} cosR_s={cos/length},\nl1={l1} L1R_s={l1/length},\nl2={l2} L2R_s={l2/length}")



slide.parquet
length=30
cos=1 cosR_s=0.03333333333333333,
l1=1 L1R_s=0.03333333333333333,
l2=1 L2R_s=0.03333333333333333

slide_mean.parquet
length=30
cos=20 cosR_s=0.6666666666666666,
l1=20 L1R_s=0.6666666666666666,
l2=20 L2R_s=0.6666666666666666

slide_statistic.parquet
length=30
cos=20 cosR_s=0.6666666666666666,
l1=20 L1R_s=0.6666666666666666,
l2=20 L2R_s=0.6666666666666666


In [21]:
slide_paths = [
   "/mnt/projects/ri_scale/privagams/breast_rgb_filter",
   "/mnt/projects/ri_scale/privagams/colon_rgb_filter",
   "/mnt/projects/ri_scale/privagams/prostate_rgb_filter",
]

slide_paths_stains = [
   "/mnt/projects/ri_scale/privagams/breast_stains",
   "/mnt/projects/ri_scale/privagams/colon_stains",
   "/mnt/projects/ri_scale/privagams/prostate_stains",
]

embeddings_files = [
   "slide.parquet",
   "slide_mean.parquet",
   "slide_statistic.parquet"
]

for file in embeddings_files:
   cos = 0
   l1 = 0
   l2 = 0
   for slide_path in slide_paths:
      cos += cos_simmilarity(slide_path, file)
      l1 += l1_simmilarity(slide_path, file)
      l2 += l2_simmilarity(slide_path, file)
   
   print()
   print(file)
   length = 30
   print(f"length={length}\ncos={cos} cosR_s={cos/length},\nl1={l1} L1R_s={l1/length},\nl2={l2} L2R_s={l2/length}")



slide.parquet
length=30
cos=16 cosR_s=0.5333333333333333,
l1=18 L1R_s=0.6,
l2=16 L2R_s=0.5333333333333333

slide_mean.parquet
length=30
cos=29 cosR_s=0.9666666666666667,
l1=29 L1R_s=0.9666666666666667,
l2=27 L2R_s=0.9

slide_statistic.parquet
length=30
cos=29 cosR_s=0.9666666666666667,
l1=29 L1R_s=0.9666666666666667,
l2=28 L2R_s=0.9333333333333333


In [ ]:
slide_path = "/mnt/projects/ri_scale/privagams/breast_rgb_filter"
print()
print(slide_path)
print()
print("#"*20 + " slide encoder " + "#"*20)
cos_simmilarity(slide_path, "slide.parquet")
l1_simmilarity(slide_path, "slide.parquet")
l2_simmilarity(slide_path, "slide.parquet")

print()
print("#"*20 + " Mean of tile embeddings " + "#"*20)
cos_simmilarity(slide_path, "slide_mean.parquet")
l1_simmilarity(slide_path, "slide_mean.parquet")
l2_simmilarity(slide_path, "slide_mean.parquet")

print()
print("#"*20 + " Mean of statistic pooling " + "#"*20)
cos_simmilarity(slide_path, "slide_statistic.parquet")
l1_simmilarity(slide_path, "slide_statistic.parquet")
l2_simmilarity(slide_path, "slide_statistic.parquet")

slide_path = "/mnt/projects/ri_scale/privagams/colon_rgb_filter"
print()
print(slide_path)
print()
print("#"*20 + " slide encoder " + "#"*20)
cos_slide += cos_simmilarity(slide_path, "slide.parquet")
l1_simmilarity(slide_path, "slide.parquet")
l2_simmilarity(slide_path, "slide.parquet")

print()
print("#"*20 + " Mean of tile embeddings " + "#"*20)
cos_simmilarity(slide_path, "slide_mean.parquet")
l1_simmilarity(slide_path, "slide_mean.parquet")
l2_simmilarity(slide_path, "slide_mean.parquet")

print()
print("#"*20 + " Mean of statistic pooling " + "#"*20)
cos_simmilarity(slide_path, "slide_statistic.parquet")
l1_simmilarity(slide_path, "slide_statistic.parquet")
l2_simmilarity(slide_path, "slide_statistic.parquet")

slide_path = "/mnt/projects/ri_scale/privagams/prostate_rgb_filter"
print()
print(slide_path)
print()
print("#"*20 + " slide encoder " + "#"*20)
cos_slide += cos_simmilarity(slide_path, "slide.parquet")
l1_simmilarity(slide_path, "slide.parquet")
l2_simmilarity(slide_path, "slide.parquet")

print()
print("#"*20 + " Mean of tile embeddings " + "#"*20)
cos_simmilarity(slide_path, "slide_mean.parquet")
l1_simmilarity(slide_path, "slide_mean.parquet")
l2_simmilarity(slide_path, "slide_mean.parquet")

print()
print("#"*20 + " Mean of statistic pooling " + "#"*20)
cos_simmilarity(slide_path, "slide_statistic.parquet")
l1_simmilarity(slide_path, "slide_statistic.parquet")
l2_simmilarity(slide_path, "slide_statistic.parquet")


/mnt/projects/ri_scale/privagams/breast_rgb_filter

#################### slide encoder ####################
Number of MIDI/FLASH matches for cos_slide.parquet is 5, R_s= 0.5
Number of MIDI/FLASH matches for L1_slide.parquet is 6, R_s= 0.6
Number of MIDI/FLASH matches for L2_slide.parquet is 5, R_s= 0.5

#################### Mean of tile embeddings ####################
Number of MIDI/FLASH matches for cos_slide_mean.parquet is 10, R_s= 1.0
Number of MIDI/FLASH matches for L1_slide_mean.parquet is 10, R_s= 1.0
Number of MIDI/FLASH matches for L2_slide_mean.parquet is 9, R_s= 0.9

#################### Mean of statistic pooling ####################
Number of MIDI/FLASH matches for cos_slide_statistic.parquet is 10, R_s= 1.0
Number of MIDI/FLASH matches for L1_slide_statistic.parquet is 10, R_s= 1.0
Number of MIDI/FLASH matches for L2_slide_statistic.parquet is 9, R_s= 0.9

/mnt/projects/ri_scale/privagams/colon_rgb_filter

#################### slide encoder ####################
Number of 